In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [3]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
for col in df.columns.to_list():
    if df[col].dtype == 'str':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0.0)

In [7]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [8]:
# Q1
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [12]:
# Q2
numerical_cols = ['annual_income','number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted']
for col in numerical_cols:
    corr = df[numerical_cols].corrwith(df[col])
    print(col, corr)

annual_income annual_income               1.000000
number_of_courses_viewed    0.161300
interaction_count           0.122842
lead_score                  0.229496
converted                   0.189652
dtype: float64
number_of_courses_viewed annual_income               0.161300
number_of_courses_viewed    1.000000
interaction_count           0.721609
lead_score                  0.757204
converted                   0.395038
dtype: float64
interaction_count annual_income               0.122842
number_of_courses_viewed    0.721609
interaction_count           1.000000
lead_score                  0.915746
converted                   0.448624
dtype: float64
lead_score annual_income               0.229496
number_of_courses_viewed    0.757204
interaction_count           0.915746
lead_score                  1.000000
converted                   0.483246
dtype: float64
converted annual_income               0.189652
number_of_courses_viewed    0.395038
interaction_count           0.448624
lead_score 

In [15]:
# Q3
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [19]:
cat_cols = ['lead_source', 'industry', 'employment_status', 'location']

from sklearn.metrics import mutual_info_score
for col in cat_cols:
    m_score = round(mutual_info_score(y_train, df_train[col]), 2)
    print(col, m_score)

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [30]:
# Q4
numerical_cols = ['annual_income','number_of_courses_viewed', 'interaction_count',
       'lead_score']
cols_to_include = cat_cols + numerical_cols

In [31]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

dv = DictVectorizer(sparse=False)



train_dict = df_train[cols_to_include].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[cols_to_include].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:,1]
converted_choice = (y_pred >= 0.5)
accuracy = (y_val == converted_choice).mean()
print("accuracy", accuracy)

accuracy 0.645


In [36]:
converted_choice

array([ True, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True, False,  True,  True,
        True,  True, False,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True, False,  True,  True, False,
        True,  True,  True,  True,  True,  True, False,  True,  True,
       False, False,  True, False,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True, False,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False, False,

In [37]:
y_val

array([1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0,
       1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0,
       1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1,
       0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0,
       1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1,

In [33]:
# Q5
features_to_remove=['lead_source',
'number_of_courses_viewed',
'interaction_count']

for fea in features_to_remove:
    current_features = [f for f in cols_to_include if f != fea]
    train_dict = df_train[current_features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[current_features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:,1]
    converted_choice = (y_pred >= 0.5)
    accuracy = (y_val == converted_choice).mean()
    print("remove feature", fea, "accuracy", accuracy)

remove feature lead_source accuracy 0.642
remove feature number_of_courses_viewed accuracy 0.643
remove feature interaction_count accuracy 0.601


In [34]:
# Q6
C = [0.000001, 0.00001, 0.0001, 0.001]
dv = DictVectorizer(sparse=False)



train_dict = df_train[cols_to_include].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[cols_to_include].to_dict(orient='records')
X_val = dv.transform(val_dict)

for c in C:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:,1]
    converted_choice = (y_pred >= 0.5)
    accuracy = (y_val == converted_choice).mean()
    print("c value", c, "accuracy", round(accuracy, 3))



c value 1e-06 accuracy 0.598
c value 1e-05 accuracy 0.598
c value 0.0001 accuracy 0.613
c value 0.001 accuracy 0.645
